# M07-01 — Parquet curated

Referencia de validación. El alumno trabaja en `notebooks/alumno/M07-01-parquet-layout.ipynb`.


## Celda 0 — localizar el repo


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
from pyspark.sql.functions import col
spark = get_spark("novashop-m07")
fact = spark.read.parquet(str(STAGING / "fact_lines"))
customers = spark.read.parquet(str(STAGING / "customers_clean"))
products = spark.read.parquet(str(STAGING / "products_clean")).dropDuplicates(["product_id"])
sales = (
    fact.join(customers, "customer_id", "inner")
    .join(products, "product_id", "left")
    .where(col("is_billable"))
    .select(
        "order_id", "order_ts", "order_month", "customer_id", "country", "segment",
        "product_id", "category", "qty", "unit_price", "discount", "gmv_line", "channel_norm",
    )
)
print(sales.count())
assert sales.count() == 1122
dest = CURATED / "sales_analytics"
CURATED.mkdir(parents=True, exist_ok=True)
sales.write.mode("overwrite").partitionBy("order_month").parquet(str(dest))
months = sorted(p.name for p in dest.iterdir() if p.is_dir() and p.name.startswith("order_month="))
print(months)
assert len(months) == 12
marzo = spark.read.parquet(str(dest)).where(col("order_month") == "2024-03")
marzo.explain("formatted")
print("marzo", marzo.count(), "total", spark.read.parquet(str(dest)).count())
assert spark.read.parquet(str(dest)).count() == 1122
print("M07-01 OK")
